# Prompt Engineering Fundamentals
## Notebook 3 — AI Engineer Practical Series

Prompt engineering is the practice of designing inputs to get reliable, 
structured, and accurate outputs from LLMs.

### What we cover:
1. Structured JSON outputs — force the model to return parseable data
2. Chain of Thought (CoT) — make the model show its reasoning
3. Few-shot prompting — teach the model with examples

***Setup***

In [ ]:
# Install required packages
# Run this cell once, then restart kernel

%pip install groq
%pip install sentence-transformers
%pip install tf-keras
%pip install numpy
%pip install python-dotenv

In [1]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = os.getenv("GROQ_MODEL")
print(f"Client ready ✅ — using {MODEL}")

Client ready ✅ — using llama-3.1-8b-instant


## 1. Structured JSON Output
Force the model to return structured data instead of free text.
This is critical in production — your application needs predictable, parseable responses.

In [3]:
def extract_structured(text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Extract information and return ONLY valid JSON.
No explanation, no markdown, no code blocks. Just raw JSON.
Format: {"name": "", "role": "", "skills": [], "experience_years": 0}"""},
            {"role": "user", "content": text}
        ]
    )
    raw = response.choices[0].message.content
    return json.loads(raw)

profile = extract_structured("""
Adi is an AI Engineer with 12 years of experience in Python, 
automation, and API integration. Previously worked in RPA.
""")

print(json.dumps(profile, indent=2))
print(f"\nName: {profile['name']}")
print(f"Skills: {profile['skills']}")

{
  "name": "Adi",
  "role": "AI Engineer",
  "skills": [
    "Python",
    "Automation",
    "API integration",
    "RPA"
  ],
  "experience_years": 12
}

Name: Adi
Skills: ['Python', 'Automation', 'API integration', 'RPA']


## 2. Chain of Thought (CoT)
Instead of asking for a direct answer, ask the model to think step by step.
Dramatically improves accuracy on reasoning, math, and multi-step problems.

In [4]:
def ask_with_cot(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """You are a precise analytical assistant.
Before answering, think through the problem step by step inside <thinking> tags.
Then give your final answer inside <answer> tags."""},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test on a reasoning problem
result = ask_with_cot("""
A company has 3 AI engineers. Each engineer can review 
5 pull requests per day. They work 5 days a week. 
How many PRs can the team review in a month (4 weeks)?
""")

print(result)

<thinking>

To find the total number of PRs the team can review in a month, we need to break down the calculation step by step.

1. Calculate the total number of PRs one engineer can review in a week:
5 PRs/day * 5 days/week = 25 PRs/week

2. Calculate the total number of PRs the team (with 3 engineers) can review in a week:
3 engineers * 25 PRs/week/engineer = 75 PRs/week

3. Calculate the total number of PRs the team can review in a month (4 weeks):
75 PRs/week * 4 weeks = 300 PRs/month

</thinking>

<answer>
The team of 3 AI engineers can review 300 PRs in a month.


## 3. Few-Shot Prompting
Provide 2-3 examples of input → output pairs before the real question.
Teaches the model your exact format and style without fine-tuning.

In [5]:
def classify_ticket(ticket):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Classify IT support tickets into: 
CRITICAL / HIGH / MEDIUM / LOW priority.
Return ONLY the priority label, nothing else."""},
            {"role": "user", "content": "Server is completely down, no one can work"},
            {"role": "assistant", "content": "CRITICAL"},
            {"role": "user", "content": "My mouse scroll wheel is a bit stiff"},
            {"role": "assistant", "content": "LOW"},
            {"role": "user", "content": "Can't access the company VPN from home"},
            {"role": "assistant", "content": "HIGH"},
            {"role": "user", "content": ticket}
        ]
    )
    return response.choices[0].message.content

# Test it
tickets = [
    "Database is throwing errors for 50% of users",
    "Need a new keyboard, mine is getting old",
    "Login page is broken, no one can sign in",
    "My monitor flickers sometimes"
]

for ticket in tickets:
    priority = classify_ticket(ticket)
    print(f"[{priority}] {ticket}")

[HIGH] Database is throwing errors for 50% of users
[LOW] Need a new keyboard, mine is getting old
[CRITICAL] Login page is broken, no one can sign in
[MEDIUM] My monitor flickers sometimes


## Summary

| Technique | When to use | Key benefit |
|---|---|---|
| Structured JSON | Any time you need parseable output | Reliable, machine-readable responses |
| Chain of Thought | Reasoning, math, multi-step problems | Dramatically improves accuracy |
| Few-Shot | Unusual formats, consistent classification | No fine-tuning needed |

These three techniques cover 80% of production prompt engineering needs.
Next: RAG Pipelines — connecting the LLM to your own documents at scale.

## ***Combine all three techniques***

In [6]:
def analyse_ticket(ticket):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Analyse IT support tickets and return ONLY valid JSON.
No explanation, no markdown. Raw JSON only.
Format: {
    "priority": "CRITICAL/HIGH/MEDIUM/LOW",
    "category": "infrastructure/access/hardware/software",
    "estimated_resolution_hours": 0,
    "reasoning": "one sentence explanation"
}"""},
            {"role": "user", "content": "Server is completely down, no one can work"},
            {"role": "assistant", "content": '{"priority": "CRITICAL", "category": "infrastructure", "estimated_resolution_hours": 2, "reasoning": "Complete outage blocking all work requires immediate response."}'},
            {"role": "user", "content": "My mouse scroll wheel is a bit stiff"},
            {"role": "assistant", "content": '{"priority": "LOW", "category": "hardware", "estimated_resolution_hours": 48, "reasoning": "Non-blocking hardware cosmetic issue."}'},
            {"role": "user", "content": ticket}
        ]
    )
    raw = response.choices[0].message.content
    return json.loads(raw)

# Test it
tickets = [
    "Database is throwing errors for 50% of users",
    "Login page is broken, no one can sign in",
    "My monitor flickers sometimes"
]

for ticket in tickets:
    result = analyse_ticket(ticket)
    print(f"\nTicket: {ticket}")
    print(json.dumps(result, indent=2))


Ticket: Database is throwing errors for 50% of users
{
  "priority": "HIGH",
  "category": "software",
  "estimated_resolution_hours": 8,
  "reasoning": "Service degradation impacting a significant number of users requires prompt resolution."
}

Ticket: Login page is broken, no one can sign in
{
  "priority": "CRITICAL",
  "category": "software",
  "estimated_resolution_hours": 1,
  "reasoning": "Blocking authentication prevents users from accessing necessary systems."
}

Ticket: My monitor flickers sometimes
{
  "priority": "MEDIUM",
  "category": "hardware",
  "estimated_resolution_hours": 4,
  "reasoning": "Intermittent hardware malfunction requires inspection and possible replacement."
}
